# Chapter 8 — Learning Signal and Ignoring Noise: Regularization, Dropout, dan Batching

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 8:** Learning Signal and Ignoring Noise: Introduction to Regularization and Batching

## Isi Chapter 8

1. Jaringan tiga lapis pada MNIST, dan mengapa hasil di data latih tidak menjamin hasil di data uji
2. Overfitting: memorization vs generalization
3. Early stopping
4. Dropout: menonaktifkan neuron secara acak
5. Batch gradient descent

## Catatan data

Notebook ini memakai dataset MNIST dari `keras.datasets`. Jalankan notebook di Google
Colab, karena dataset MNIST otomatis tersedia di sana. Setiap sel training memakai
data yang sama dengan chapter ini: 1.000 gambar latih dan 10.000 gambar uji.

Notebook disimpan tanpa output yang sudah dieksekusi. Output akan muncul ketika
notebook dijalankan di Colab.

## 1. Jaringan tiga lapis pada MNIST

Saya memuat MNIST, mengubah setiap gambar menjadi vektor 784 nilai (28 x 28), dan
mengubah label menjadi *one-hot* dengan 10 kolom.

In [1]:
import sys
import numpy as np
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 1.000 gambar pertama untuk training, dinormalisasi ke rentang 0-1
images = x_train[0:1000].reshape(1000, 28 * 28) / 255
labels_raw = y_train[0:1000]

one_hot_labels = np.zeros((len(labels_raw), 10))
for i, l in enumerate(labels_raw):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

# Data uji: seluruh 10.000 gambar test
test_images = x_test.reshape(len(x_test), 28 * 28) / 255
test_labels = np.zeros((len(y_test), 10))
for i, l in enumerate(y_test):
    test_labels[i][l] = 1

print('images      :', images.shape)
print('labels      :', labels.shape)
print('test_images :', test_images.shape)
print('test_labels :', test_labels.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
images      : (1000, 784)
labels      : (1000, 10)
test_images : (10000, 784)
test_labels : (10000, 10)


Jaringan yang saya pakai: tiga lapis dengan relu di lapisan tersembunyi (40 node),
learning rate `0.005`, dan 350 iterasi. Pertama, saya melatih jaringan hanya pada
1.000 gambar latih untuk melihat apakah jaringan bisa belajar memprediksi data itu.

In [2]:
np.random.seed(1)
relu = lambda x: (x >= 0) * x
relu2deriv = lambda x: x >= 0

alpha, iterations, hidden_size, pixels_per_image, num_labels = (0.005, 350, 40, 784, 10)

weights_0_1 = 0.2 * np.random.random((pixels_per_image, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)

        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i+1]))

        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if j % 50 == 0 or j == iterations - 1:
        print('I:', j,
              '| Error:', round(error / float(len(images)), 3),
              '| Correct:', round(correct_cnt / float(len(images)), 3))

I: 0 | Error: 0.722 | Correct: 0.537
I: 50 | Error: 0.204 | Correct: 0.966
I: 100 | Error: 0.167 | Correct: 0.984
I: 150 | Error: 0.145 | Correct: 0.991
I: 200 | Error: 0.13 | Correct: 0.998
I: 250 | Error: 0.12 | Correct: 0.999
I: 300 | Error: 0.113 | Correct: 0.999
I: 349 | Error: 0.109 | Correct: 1.0


Jika training berjalan seperti yang diharapkan, akurasi latih pada iterasi terakhir
mendekati 1.0. Ini kelihatannya bagus, tetapi pertanyaan yang lebih penting: seberapa
baik jaringan ini pada gambar yang belum pernah dilihatnya? Saya menguji dengan 10.000
gambar uji:

In [3]:
test_error, test_correct_cnt = (0.0, 0)
for i in range(len(test_images)):
    layer_0 = test_images[i:i+1]
    layer_1 = relu(np.dot(layer_0, weights_0_1))
    layer_2 = np.dot(layer_1, weights_1_2)
    test_error += np.sum((test_labels[i:i+1] - layer_2) ** 2)
    test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))

print('Test-Err:', round(test_error / float(len(test_images)), 3),
      '| Test-Acc:', round(test_correct_cnt / float(len(test_images)), 4))

Test-Err: 0.653 | Test-Acc: 0.7073


Jika akurasi uji jauh lebih rendah daripada akurasi latih, berarti jaringan hanya
menghafal 1.000 gambar latih dan tidak menggeneralisasi dengan baik. Akurasi uji adalah
ukuran yang penting, karena itu yang mencerminkan performa jaringan di dunia nyata.

## 2. Memorization vs generalization

Jaringan menyesuaikan setiap bobot agar prediksi untuk input spesifik menjadi benar.
Karena itu, jaringan yang sudah dilatih hanya akan bekerja baik untuk gambar yang
sangat mirip dengan gambar latih. Jika inputnya terlihat asing, prediksinya cenderung
acak.

Yang saya perlu pantau adalah akurasi uji selama training, bukan hanya akurasi latih.
Saya melatih ulang jaringan dan mencatat akurasi latih dan uji setiap 10 iterasi:

In [4]:
np.random.seed(1)
weights_0_1 = 0.2 * np.random.random((pixels_per_image, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)
        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i+1]))
        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if j % 10 == 0 or j == iterations - 1:
        test_error, test_correct_cnt = (0.0, 0)
        for i in range(len(test_images)):
            layer_0 = test_images[i:i+1]
            layer_1 = relu(np.dot(layer_0, weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)
            test_error += np.sum((test_labels[i:i+1] - layer_2) ** 2)
            test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))
        print('I:', j,
              '| Train-Err:', round(error / float(len(images)), 3),
              '| Train-Acc:', round(correct_cnt / float(len(images)), 3),
              '| Test-Err:', round(test_error / float(len(test_images)), 3),
              '| Test-Acc:', round(test_correct_cnt / float(len(test_images)), 4))

I: 0 | Train-Err: 0.722 | Train-Acc: 0.537 | Test-Err: 0.602 | Test-Acc: 0.6488
I: 10 | Train-Err: 0.312 | Train-Acc: 0.901 | Test-Err: 0.42 | Test-Acc: 0.8114
I: 20 | Train-Err: 0.261 | Train-Acc: 0.93 | Test-Err: 0.414 | Test-Acc: 0.8111
I: 30 | Train-Err: 0.233 | Train-Acc: 0.946 | Test-Err: 0.417 | Test-Acc: 0.8066
I: 40 | Train-Err: 0.216 | Train-Acc: 0.956 | Test-Err: 0.427 | Test-Acc: 0.8019
I: 50 | Train-Err: 0.204 | Train-Acc: 0.966 | Test-Err: 0.438 | Test-Acc: 0.7982
I: 60 | Train-Err: 0.195 | Train-Acc: 0.967 | Test-Err: 0.448 | Test-Acc: 0.7921
I: 70 | Train-Err: 0.187 | Train-Acc: 0.975 | Test-Err: 0.458 | Test-Acc: 0.7864
I: 80 | Train-Err: 0.18 | Train-Acc: 0.979 | Test-Err: 0.467 | Test-Acc: 0.7817
I: 90 | Train-Err: 0.173 | Train-Acc: 0.981 | Test-Err: 0.474 | Test-Acc: 0.7758
I: 100 | Train-Err: 0.167 | Train-Acc: 0.984 | Test-Err: 0.482 | Test-Acc: 0.7706
I: 110 | Train-Err: 0.162 | Train-Acc: 0.984 | Test-Err: 0.49 | Test-Acc: 0.7686
I: 120 | Train-Err: 0.158 | Tra

Pola yang perlu diperhatikan dari log ini: jika akurasi uji naik di awal lalu perlahan
turun, sementara akurasi latih terus naik, itu tanda **overfitting**.

## 3. Overfitting dan early stopping

Overfitting berarti jaringan belajar noise dalam dataset, bukan sinyal yang
sebenarnya. Analogi cetakan garpu membantu saya memahaminya: cetakan yang dibuat
hanya dari garpu tiga cabang akan menolak garpu empat cabang, walaupun keduanya
tetap garpu.

Cara paling sederhana untuk menghindari overfitting adalah **early stopping**:
hentikan training ketika akurasi uji mulai memburuk. Early stopping adalah cara paling
murah untuk regularization, tetapi harus dipilih menggunakan **validation set**, bukan
test set. Jika test set dipakai untuk memilih kapan berhenti, saya bisa overfit ke test
set itu sendiri.

Regularization adalah kelompok metode untuk membuat model menggeneralisasi ke data baru.
Caranya biasanya dengan membuat model sulit mempelajari detail halus dari data latih.

## 4. Dropout: menonaktifkan neuron secara acak

Dropout bekerja dengan cara sederhana: selama training, sebagian neuron di lapisan
tersembunyi diatur menjadi 0 secara acak. Setiap iterasi, jaringan hanya belajar
melalui subset neuron yang berbeda.

Alasan dropout bekerja: jaringan kecil punya kapasitas lebih rendah, sehingga lebih
sulit menghafal noise. Dropout membuat jaringan besar bertindak seperti banyak jaringan
kecil. Jika saya melatih banyak jaringan dan rata-ratakan hasilnya (ensembling), noise
dari setiap jaringan akan saling meniadakan, dan yang tersisa adalah sinyal bersama.

Implementasi di kode: saya membuat `dropout_mask` berisi 0 dan 1 secara acak untuk
`layer_1`. Hasil `layer_1` dikali mask lalu dikali 2. Pengali 2 dipakai karena separuh
neuron dimatikan, sehingga jumlah sinyal yang diterima `layer_2` dijaga tetap sama.
Gradien di lapisan tersembunyi juga dikali mask yang sama.

Saya menjalankan training dengan dropout. Jaringan tersembunyinya saya perbesar menjadi
100 node karena dropout membuatnya mudah menghafal lebih sedikit, dan kita ingin
kapasitasnya cukup:

In [5]:
np.random.seed(1)

alpha, iterations, hidden_size = (0.005, 300, 100)
pixels_per_image, num_labels = (784, 10)

weights_0_1 = 0.2 * np.random.random((pixels_per_image, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2

        layer_2 = np.dot(layer_1, weights_1_2)
        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i+1]))

        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        layer_1_delta *= dropout_mask
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if j % 10 == 0:
        test_error, test_correct_cnt = (0.0, 0)
        for i in range(len(test_images)):
            layer_0 = test_images[i:i+1]
            layer_1 = relu(np.dot(layer_0, weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)
            test_error += np.sum((test_labels[i:i+1] - layer_2) ** 2)
            test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))
        print('I:', j,
              '| Test-Err:', round(test_error / float(len(test_images)), 3),
              '| Test-Acc:', round(test_correct_cnt / float(len(test_images)), 4),
              '| Train-Err:', round(error / float(len(images)), 3),
              '| Train-Acc:', round(correct_cnt / float(len(images)), 3))

I: 0 | Test-Err: 0.641 | Test-Acc: 0.6333 | Train-Err: 0.891 | Train-Acc: 0.413
I: 10 | Test-Err: 0.459 | Test-Acc: 0.787 | Train-Err: 0.472 | Train-Acc: 0.764
I: 20 | Test-Err: 0.416 | Test-Acc: 0.8133 | Train-Err: 0.431 | Train-Acc: 0.809
I: 30 | Test-Err: 0.421 | Test-Acc: 0.8114 | Train-Err: 0.416 | Train-Acc: 0.811
I: 40 | Test-Err: 0.42 | Test-Acc: 0.8112 | Train-Err: 0.413 | Train-Acc: 0.827
I: 50 | Test-Err: 0.41 | Test-Acc: 0.8133 | Train-Err: 0.392 | Train-Acc: 0.836
I: 60 | Test-Err: 0.413 | Test-Acc: 0.8236 | Train-Err: 0.402 | Train-Acc: 0.836
I: 70 | Test-Err: 0.412 | Test-Acc: 0.8033 | Train-Err: 0.383 | Train-Acc: 0.857
I: 80 | Test-Err: 0.411 | Test-Acc: 0.8054 | Train-Err: 0.387 | Train-Acc: 0.854
I: 90 | Test-Err: 0.411 | Test-Acc: 0.8144 | Train-Err: 0.377 | Train-Acc: 0.868
I: 100 | Test-Err: 0.411 | Test-Acc: 0.7903 | Train-Err: 0.37 | Train-Acc: 0.864
I: 110 | Test-Err: 0.411 | Test-Acc: 0.8003 | Train-Err: 0.372 | Train-Acc: 0.868
I: 120 | Test-Err: 0.403 | Test

Dengan dropout, akurasi latih diharapkan naik lebih lambat dan tidak langsung mencapai
1.0. Yang saya cari dari log ini adalah selisih antara akurasi latih dan uji: jika
selisihnya lebih kecil daripada pada percobaan tanpa dropout, dropout membantu
mengurangi overfitting. Dropout membuat training lebih sulit, tetapi jaringan yang
dihasilkan lebih tahan terhadap overfitting.

## 5. Batch gradient descent

Batch gradient descent memproses sejumlah contoh sekaligus (misalnya 100), lalu
merata-ratakan pembaruan bobotnya. Hasilnya:

- Perhitungan lebih cepat, karena `np.dot` memproses 100 vektor sekaligus.
- Pembaruan bobot lebih halus, karena rata-rata dari banyak contoh lebih stabil daripada satu contoh.
- `alpha` yang dipakai di sini adalah `0.001`. Nilai ini dipilih bersama `batch_size`, dan pasangan keduanya perlu dicoba sendiri.

Ukuran batch yang umum berkisar antara 8 sampai 256. Saya mencoba `batch_size = 100`:

In [6]:
np.random.seed(1)

batch_size = 100
alpha, iterations = (0.001, 300)
pixels_per_image, num_labels, hidden_size = (784, 10, 100)

weights_0_1 = 0.2 * np.random.random((pixels_per_image, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = (i * batch_size), ((i + 1) * batch_size)

        layer_0 = images[batch_start:batch_end]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = np.dot(layer_1, weights_1_2)

        error += np.sum((labels[batch_start:batch_end] - layer_2) ** 2)
        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) ==
                               np.argmax(labels[batch_start+k:batch_start+k+1]))

        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / batch_size
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        layer_1_delta *= dropout_mask
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if j % 10 == 0:
        test_error, test_correct_cnt = (0.0, 0)
        for i in range(len(test_images)):
            layer_0 = test_images[i:i+1]
            layer_1 = relu(np.dot(layer_0, weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)
            test_error += np.sum((test_labels[i:i+1] - layer_2) ** 2)
            test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))
        print('I:', j,
              '| Test-Err:', round(test_error / float(len(test_images)), 3),
              '| Test-Acc:', round(test_correct_cnt / float(len(test_images)), 4),
              '| Train-Err:', round(error / float(len(images)), 3),
              '| Train-Acc:', round(correct_cnt / float(len(images)), 3))

I: 0 | Test-Err: 1.514 | Test-Acc: 0.0667 | Train-Err: 2.08 | Train-Acc: 0.084
I: 10 | Test-Err: 1.108 | Test-Acc: 0.118 | Train-Err: 1.534 | Train-Acc: 0.097
I: 20 | Test-Err: 1.021 | Test-Acc: 0.1621 | Train-Err: 1.348 | Train-Acc: 0.147
I: 30 | Test-Err: 0.974 | Test-Acc: 0.1973 | Train-Err: 1.269 | Train-Acc: 0.147
I: 40 | Test-Err: 0.939 | Test-Acc: 0.2319 | Train-Err: 1.226 | Train-Acc: 0.141
I: 50 | Test-Err: 0.91 | Test-Acc: 0.2609 | Train-Err: 1.128 | Train-Acc: 0.19
I: 60 | Test-Err: 0.885 | Test-Acc: 0.2943 | Train-Err: 1.103 | Train-Acc: 0.213
I: 70 | Test-Err: 0.864 | Test-Acc: 0.3233 | Train-Err: 1.063 | Train-Acc: 0.221
I: 80 | Test-Err: 0.845 | Test-Acc: 0.3509 | Train-Err: 1.028 | Train-Acc: 0.253
I: 90 | Test-Err: 0.828 | Test-Acc: 0.3755 | Train-Err: 0.993 | Train-Acc: 0.271
I: 100 | Test-Err: 0.814 | Test-Acc: 0.3953 | Train-Err: 0.976 | Train-Acc: 0.28
I: 110 | Test-Err: 0.801 | Test-Acc: 0.4166 | Train-Err: 0.938 | Train-Acc: 0.312
I: 120 | Test-Err: 0.79 | Test-A

Pada batch training, saya mengharapkan kurva akurasi latih terlihat lebih halus
dibanding training satu contoh per update. Ini masuk akal, karena setiap pembaruan
bobot adalah rata-rata dari 100 contoh, sehingga noise dari satu contoh tidak terlalu
berpengaruh.

## Key Ideas — Chapter 8

- Akurasi latih yang tinggi tidak menjamin model bekerja pada data baru. Akurasi uji adalah ukuran yang penting.
- Overfitting terjadi ketika model mempelajari noise dan kehilangan kemampuan generalisasi.
- Early stopping menghentikan training ketika akurasi uji mulai turun. Pilihannya harus didasarkan pada validation set.
- Dropout mematikan neuron secara acak selama training, lalu mengalikan output dengan 2 agar skalanya tetap sama.
- Batch gradient descent merata-ratakan pembaruan dari banyak contoh, sehingga training lebih cepat dan lebih stabil.

## Kesimpulan

Chapter 8 menunjukkan bahwa jaringan yang mampu belajar dengan sempurna di data latih
belum tentu bisa menggeneralisasi. Regularization seperti early stopping dan dropout
membantu jaringan mempelajari sinyal dan mengabaikan noise. Batching membuat training
lebih cepat dan lebih stabil.